# Notebook 3: Customer Segmentation & Health Scoring

---

## Business Question

**Can we group customers into meaningful segments and identify at-risk customers before they churn?**

In Notebook 2, I identified key churn drivers. Now I want to go further - creating customer segments for targeted action, and building a health score system that can flag at-risk customers proactively.

## Why This Matters

Different customers have different needs. Treating everyone the same is inefficient. Segmentation lets us:
- Target retention efforts at the right people
- Allocate resources based on customer value
- Personalize communications and offers

A health score gives operations teams a simple tool to prioritize outreach.

## Hypothesis

Customers naturally cluster into distinct groups based on behavior and value. A health score based on engagement signals can predict churn risk.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../data/prepared_data.csv')
print(f"Loaded {len(df):,} customers")

---

## Feature Selection for Clustering

I need to pick features that differentiate customers meaningfully. I'll focus on behavioral and value metrics, not demographics. Demographics can be useful for targeting, but behavior is what drives churn.

---

In [ ]:
# Features for clustering
cluster_features = ['watch_hours', 'last_login_days', 'monthly_fee', 
                   'number_of_profiles', 'avg_watch_time_per_day']

X = df[cluster_features].copy()

print("Features selected for clustering:")
for f in cluster_features:
    print(f"  - {f}")

print(f"\nChecking for nulls: {X.isnull().sum().sum()}")
print(f"Shape: {X.shape}")

In [ ]:
# Scale features - essential for K-Means since it's distance-based
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled to mean=0, std=1")

---

## Determining Optimal Number of Clusters

I'll use both the elbow method and silhouette score. But here's the thing - in business contexts, I also need to consider: **Can I actually explain these segments to stakeholders?** More than 5-6 segments become hard to operationalize.

---

In [ ]:
# Test k=2 to k=10
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow method
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4 chosen')
axes[0].legend()

# Silhouette scores
axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score', fontsize=14, fontweight='bold')
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4 chosen')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Detailed silhouette scores
print("Silhouette scores by k:")
for k, s in zip(K_range, silhouettes):
    marker = " <-- chosen" if k == 4 else ""
    print(f"  k={k}: {s:.3f}{marker}")

### Decision: k=4

**Reasoning:**
- Silhouette score is decent at k=4
- Elbow curve shows diminishing returns after k=4
- **Business interpretability:** 4 segments are manageable for marketing/CS teams
- More clusters would fragment the customer base too much

**Tradeoff:** Higher k might capture more nuance, but at the cost of operational complexity. I'd rather have 4 actionable segments than 8 segments that nobody can remember.

---

## Building the Segments

---

In [ ]:
# Fit K-Means with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['segment'] = kmeans.fit_predict(X_scaled)

print("Segment distribution:")
print(df['segment'].value_counts().sort_index())

In [ ]:
# Analyze segment profiles
segment_profiles = df.groupby('segment').agg({
    'watch_hours': 'mean',
    'last_login_days': 'mean',
    'monthly_fee': 'mean',
    'number_of_profiles': 'mean',
    'avg_watch_time_per_day': 'mean',
    'churned': ['mean', 'count']
}).round(2)

segment_profiles.columns = ['avg_watch', 'avg_login_days', 'avg_fee', 'avg_profiles', 
                            'avg_daily_watch', 'churn_rate', 'customers']
segment_profiles['churn_rate'] = (segment_profiles['churn_rate'] * 100).round(1)
segment_profiles['pct_total'] = (segment_profiles['customers'] / len(df) * 100).round(1)

print("Segment Profiles:")
print(segment_profiles)

---

## Naming the Segments

Based on the profiles, I can interpret what each segment represents and give them business-friendly names.

---

In [ ]:
# Map segments to names based on behavior
segment_names = {}

for seg in segment_profiles.index:
    watch = segment_profiles.loc[seg, 'avg_watch']
    login = segment_profiles.loc[seg, 'avg_login_days']
    fee = segment_profiles.loc[seg, 'avg_fee']
    churn = segment_profiles.loc[seg, 'churn_rate']
    
    # Decision logic
    if watch > 25 and login < 15:
        segment_names[seg] = 'Power Users'
    elif login > 35 or churn > 35:
        segment_names[seg] = 'At-Risk'
    elif fee >= 15:
        segment_names[seg] = 'Premium'
    else:
        segment_names[seg] = 'Casual'

df['segment_name'] = df['segment'].map(segment_names)

print("Segment Names:")
for seg, name in segment_names.items():
    churn = segment_profiles.loc[seg, 'churn_rate']
    print(f"  Segment {seg}: {name} (churn rate: {churn}%)")

In [ ]:
# Final segment summary with business metrics
final_segments = df.groupby('segment_name').agg({
    'customer_id': 'count',
    'churned': 'mean',
    'watch_hours': 'mean',
    'last_login_days': 'mean',
    'monthly_fee': ['mean', 'sum'],
    'clv_estimate': 'mean'
}).round(2)

final_segments.columns = ['customers', 'churn_rate', 'avg_watch', 'avg_login_days', 
                          'avg_fee', 'total_revenue', 'avg_clv']
final_segments['churn_rate'] = (final_segments['churn_rate'] * 100).round(1)
final_segments['pct_total'] = (final_segments['customers'] / len(df) * 100).round(1)

print("\nFinal Segment Summary:")
print(final_segments)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Customer distribution
final_segments['customers'].plot(kind='bar', ax=axes[0, 0], color='#3498db', alpha=0.8)
axes[0, 0].set_title('Customers by Segment', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Number of Customers')
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45)

# Churn rate
final_segments['churn_rate'].sort_values(ascending=False).plot(kind='bar', ax=axes[0, 1], color='#e74c3c', alpha=0.8)
axes[0, 1].set_title('Churn Rate by Segment', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Churn Rate (%)')
axes[0, 1].axhline(y=df['churned'].mean()*100, color='gray', linestyle='--')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45)

# Revenue
final_segments['total_revenue'].plot(kind='bar', ax=axes[1, 0], color='#27ae60', alpha=0.8)
axes[1, 0].set_title('Total Revenue by Segment', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Monthly Revenue ($)')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)

# CLV
final_segments['avg_clv'].plot(kind='bar', ax=axes[1, 1], color='#9b59b6', alpha=0.8)
axes[1, 1].set_title('Average Customer Lifetime Value', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Estimated CLV ($)')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

### Segment Interpretation

| Segment | Characteristics | Churn Risk | Strategy |
|---------|----------------|------------|----------|
| **Power Users** | High watch hours, recent activity | Low | Retain with exclusives, convert to advocates |
| **Premium** | Higher tier, moderate engagement | Medium | Upsell family plans, add value |
| **Casual** | Moderate engagement, lower tier | Medium | Increase engagement, personalized content |
| **At-Risk** | Low engagement, high login recency | High | Immediate intervention, win-back campaigns |

---

## Building a Customer Health Score

Beyond segments, I want a continuous health score (0-100) that can identify at-risk customers within any segment. This is more actionable for day-to-day operations - CS teams can prioritize outreach based on health scores.

---

In [ ]:
def calculate_health_score(row):
    """
    Health score from 0-100. Higher = healthier customer.
    
    Factor weights based on correlation analysis from Notebook 2:
    - Login recency: 40% (strongest predictor)
    - Watch hours: 30% (engagement indicator)
    - Subscription tier: 15% (value indicator)
    - Number of profiles: 15% (commitment indicator)
    """
    score = 0
    
    # Login recency (0-40 points)
    if row['last_login_days'] <= 7:
        score += 40
    elif row['last_login_days'] <= 14:
        score += 32
    elif row['last_login_days'] <= 21:
        score += 24
    elif row['last_login_days'] <= 30:
        score += 16
    elif row['last_login_days'] <= 45:
        score += 8
    # 60+ days = 0 points
    
    # Watch hours (0-30 points)
    if row['watch_hours'] >= 30:
        score += 30
    elif row['watch_hours'] >= 20:
        score += 24
    elif row['watch_hours'] >= 10:
        score += 18
    elif row['watch_hours'] >= 5:
        score += 12
    else:
        score += 6
    
    # Subscription tier (0-15 points)
    if row['subscription_type'] == 'Premium':
        score += 15
    elif row['subscription_type'] == 'Standard':
        score += 10
    else:
        score += 5
    
    # Profiles (0-15 points) - more profiles = more invested
    if row['number_of_profiles'] >= 4:
        score += 15
    elif row['number_of_profiles'] >= 3:
        score += 12
    elif row['number_of_profiles'] >= 2:
        score += 8
    else:
        score += 5
    
    return score

df['health_score'] = df.apply(calculate_health_score, axis=1)

print("Health Score Distribution:")
print(df['health_score'].describe().round(1))

In [ ]:
# Categorize health scores
def health_category(score):
    if score >= 80:
        return 'Healthy'
    elif score >= 60:
        return 'Monitor'
    elif score >= 40:
        return 'At-Risk'
    else:
        return 'Critical'

df['health_category'] = df['health_score'].apply(health_category)

# Validate: does health score actually predict churn?
health_churn = df.groupby('health_category').agg({
    'churned': ['mean', 'count'],
    'monthly_fee': 'sum'
})

health_churn.columns = ['actual_churn_rate', 'customers', 'revenue']
health_churn['actual_churn_rate'] = (health_churn['actual_churn_rate'] * 100).round(1)

print("Validation: Actual Churn by Health Category")
print(health_churn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Health score distribution
axes[0].hist(df['health_score'], bins=20, color='#3498db', edgecolor='white', alpha=0.7)
axes[0].axvline(df['health_score'].median(), color='red', linestyle='--', 
                label=f'Median: {df["health_score"].median():.0f}')
axes[0].set_title('Customer Health Score Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Health Score (0-100)')
axes[0].set_ylabel('Number of Customers')
axes[0].legend()

# Churn by health category
health_order = ['Healthy', 'Monitor', 'At-Risk', 'Critical']
health_churn_ordered = health_churn.reindex(health_order)
colors = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c']
health_churn_ordered['actual_churn_rate'].plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Actual Churn Rate by Health Category', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Actual Churn Rate (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

### Validation: Health Score Works

**The health score successfully predicts churn.** Critical category has the highest actual churn rate, Healthy has the lowest. This validates the scoring logic.

**Business Application:** Customer Success teams can use health scores to:
- Prioritize outreach to At-Risk and Critical customers
- Set up automated alerts when health scores drop
- Track cohort health over time

---

## Segment × Health Cross-Analysis

How do health categories distribute across segments?

---

In [ ]:
# Cross-tabulation
segment_health = pd.crosstab(df['segment_name'], df['health_category'], normalize='index') * 100
segment_health = segment_health[['Healthy', 'Monitor', 'At-Risk', 'Critical']].round(1)

print("Health Distribution by Segment (%):")
print(segment_health)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c']
segment_health.plot(kind='bar', stacked=True, ax=ax, color=colors, edgecolor='white')

ax.set_title('Health Category Distribution by Segment', fontsize=14, fontweight='bold')
ax.set_ylabel('Percentage')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.legend(title='Health Category', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

---

## Customer Lifetime Value Estimation

Let me estimate CLV more sophisticatedly, incorporating segment-specific retention rates.

---

In [ ]:
# Calculate segment-specific retention rates
segment_retention = 1 - df.groupby('segment_name')['churned'].mean()

# Map retention rate to each customer
df['segment_retention_rate'] = df['segment_name'].map(segment_retention)

# Estimate customer lifetime (months)
# Using formula: lifetime = 1 / churn_rate
# Cap at 36 months (3 years) for stability
df['estimated_lifetime_months'] = 1 / (1 - df['segment_retention_rate'] + 0.01)
df['estimated_lifetime_months'] = df['estimated_lifetime_months'].clip(upper=36)

# CLV = monthly fee × estimated lifetime
df['clv_calculated'] = df['monthly_fee'] * df['estimated_lifetime_months']

# Segment CLV summary
clv_summary = df.groupby('segment_name').agg({
    'clv_calculated': ['mean', 'sum'],
    'customer_id': 'count'
}).round(2)

clv_summary.columns = ['avg_clv', 'total_clv', 'customers']
print("Customer Lifetime Value by Segment:")
print(clv_summary)

---

## Save Segmented Data

---

In [ ]:
df.to_csv('../data/segmented_data.csv', index=False)

print(f"Segmented data saved: {df.shape}")
print(f"\nNew features added:")
print(f"  - segment (0-3)")
print(f"  - segment_name (Power Users, Premium, Casual, At-Risk)")
print(f"  - health_score (0-100)")
print(f"  - health_category (Healthy, Monitor, At-Risk, Critical)")
print(f"  - clv_calculated")

---

## Summary

### What I Built

1. **4 Customer Segments:** Power Users, Premium, Casual, At-Risk
2. **Health Score System:** 0-100 scale with 4 categories
3. **CLV Estimates:** Segment-specific lifetime value calculations

### Key Findings

- **At-Risk segment** has the highest churn rate and needs immediate attention
- **Power Users** are our most valuable customers with lowest churn
- **Health score validates well** - Critical category has significantly higher actual churn

### Limitations

- K-Means assumes spherical clusters; real customer behavior might be more complex
- Health score weights are heuristic; could be optimized with ML
- CLV is simplified; doesn't account for acquisition costs or discount rates

### Next Step

In Notebook 4, I'll build machine learning models to predict churn with more precision, and compare them against the heuristic health score.

---